In [1]:
import os
import sys
import torch
import tiktoken

project_root = os.path.dirname(os.path.abspath("")) # since notebook is in evaluation/
sys.path.insert(0, project_root)

import model
from model.model import GPT, GPTConfig
model.GPTConfig = GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [3]:
# Load configuration matching the notebook setup
config = GPTConfig(
    block_size=256,
    vocab_size=50257,
    n_layer=8,
    n_head=1,
    n_embd=128,
    dropout=0.1
)

# Initialize model
model = GPT(config)
model.to(device)
model.eval()

model_path = os.path.join(project_root, "training", "nanogpt_checkpoint_1.pt")
if not os.path.exists(model_path):
    print(f"Error: {model_path} not found.")
    print("Please run the notebook 'training/training_pipeline.ipynb' to train and save the model.")
else:
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")

number of parameters: 8.02M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_1.pt


In [6]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]\nHi, kids. Do you like violence?"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
Hi, kids. Do you like violence?'



In [7]:
print("Temperature Sampling")
# Setting manual seed for determinism in generation examples
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, temperature=0.7, repetition_penalty=1.2)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
Hi, kids. Do you like violence? You are for real"
(I'm a bad side)
My way, my way was over (Yeah!)
These days I was playin' mad enough to win some more chance
One time is four degrees and three, everybody know that we came in the mind
And you catch your face top off the door
The way we're the one of you
Now you can go back (It's right)
Whole time we need two times
You got it down before I feel like the same (That shit?)
All them hours with you, now we get up on the same way
Lot of me so many
But I don't mean, they've been ballin' out this, they know what we do is (Whoa?)
We ain't hard when we rollin', he still stressin', "What she want?" (He's right now), just how I am you feel like a fouch, bitch
Back in fire with the beat, holders at the box models
Just though, it make me grab all these bitches up
Bent on my hood, you say it and whatever the fuck on my chin
Shout y'all hit the club, 